In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer


In [2]:
#从预训练的模型里面加载参数
model = AutoModelForCausalLM.from_pretrained(
    'Qwen2.5-0.5B-Instruct',
    torch_dtype='auto'
)
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [3]:
#加载token模型
tokenizer = AutoTokenizer.from_pretrained('Qwen2.5-0.5B-Instruct')

In [4]:
#给大模型的输入;system是全局设定;user给大模型的问题
message = [
    {'role': 'system', 'content': '你是千问，一个有用的人工智能。'},
    {'role': 'user', 'content': '我爱北京'}
]

In [5]:
#给上面的message加标记;最后一行<|im_start|>assistant告诉大模型可以回答问题了
text = tokenizer.apply_chat_template(
    message,
    tokenize=False,
    add_generation_prompt=True
)
#print(text)

In [6]:
#把text变成token,即数字;attention_mask全是1表示大模型可以看到全部的输入
model_inputs = tokenizer([text], return_tensors='pt').to('cpu')
print(model_inputs)

{'input_ids': tensor([[151644,   8948,    198, 105043,  99320,  56007,   3837,  46944, 115404,
         100623,  48692, 100168,   1773, 151645,    198, 151644,    872,    198,
          35946,  99242,  68990, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}


In [7]:
#大模型生成回答,要等待运行一段时间
generated_ids = model.generate(**model_inputs, max_new_tokens=512)
print(generated_ids)

tensor([[151644,   8948,    198, 105043,  99320,  56007,   3837,  46944, 115404,
         100623,  48692, 100168,   1773, 151645,    198, 151644,    872,    198,
          35946,  99242,  68990, 151645,    198, 151644,  77091,    198, 112169,
          87026,  99729,  68990,   6313, 106870, 110117,  86119,  57191,  85106,
         100364,  37945, 102422, 106525,   1773,  68990, 115164, 102216, 104523,
          57218, 102550, 105961,   3837, 102215, 100022,  99470,  99993,   5373,
         100390,  99602,  99998, 104365,  99348,   3837, 104297, 111404, 105322,
         103092, 107581, 101442, 102550,   1773,  99880,  87026, 105081,   9370,
         102275,  15946, 100006, 104619, 103444, 107232,  33108, 106369, 104843,
           1773, 151645]])


In [11]:
#把输入部分去掉
generated_ids = [output_ids[len(input_ids):]
                            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
print(generated_ids)

[tensor([112169, 104188,  87026,  32664,  68990,  18830, 102721, 110325,   6313,
         68990, 115164, 102216, 100022,  57218, 100390, 116794, 105961,   3837,
        104012, 104653, 108077,  33108, 105664,  99795, 106686,   1773, 102215,
        109462,   9370, 100386,  23836,   5373, 116498,   9370,  99470,  99993,
         99998, 104437,   9370, 116010, 106272,   3837,  71268, 103973, 104048,
        109010,  33108, 109011,   3407, 106870,  85106, 101888,  68990, 105427,
        100631, 110117,  86119,  37945, 102422, 106525,   3837, 105351, 110121,
        113445, 100364,   1773, 105081,  99424,  57191, 102275,  13343,   3837,
        101360, 104028, 104365,   5373,  99348, 101904, 101034, 101964, 103092,
        107581, 104367, 107877,   1773,  99880,  87026,  18493, 106114,  68990,
        106517, 104383, 107071,  68536, 110586,   9370, 105129,   6313, 151645])]


In [8]:
#把数字转换成文字
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [9]:
print(response)

system
你是千问，一个有用的人工智能。
user
我爱北京
assistant
很高兴您喜欢北京！如果您有任何问题或需要帮助，请随时告诉我。北京是一座充满活力与魅力的城市，无论是历史古迹、现代科技还是美食文化，都能让您感受到这座城市的独特魅力。希望您在北京的旅行中能够收获满满的知识和美好的回忆。


In [14]:
!pip install datasets trl peft # 安装需要等待一段时间会看到输出

Looking in indexes: http://mirrors.aliyun.com/pypi/simple/
     ---------------------------------------- 0.0/25.1 MB ? eta -:--:--
     -- ------------------------------------- 1.3/25.1 MB 13.4 MB/s eta 0:00:02
     --- ------------------------------------ 2.1/25.1 MB 6.5 MB/s eta 0:00:04
     ---- ----------------------------------- 2.9/25.1 MB 5.4 MB/s eta 0:00:05
     ----- ---------------------------------- 3.7/25.1 MB 4.8 MB/s eta 0:00:05
     ------- -------------------------------- 4.5/25.1 MB 4.4 MB/s eta 0:00:05
     -------- ------------------------------- 5.2/25.1 MB 4.5 MB/s eta 0:00:05
     --------- ------------------------------ 6.0/25.1 MB 4.3 MB/s eta 0:00:05
     ---------- ----------------------------- 6.8/25.1 MB 4.2 MB/s eta 0:00:05
     ------------ --------------------------- 7.6/25.1 MB 4.2 MB/s eta 0:00:05
     ------------- -------------------------- 8.7/25.1 MB 4.1 MB/s eta 0:00:04
     -------------- ------------------------- 9.2/25.1 MB 4.1 MB/s eta 0:00:04

In [11]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

dataset = load_dataset("tatsu-lab-alpaca", split="train")

peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [13]:
trainer = SFTTrainer(
    'Qwen2.5-0.5B-Instruct',
    train_dataset=dataset,
    args=SFTConfig(output_dir="tmp"),
    peft_config=peft_config,
    dataset_text_field='instruction'
)

d:\ProgramData\anaconda3\envs\llms\Lib\site-packages\trl\trainer\sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
d:\ProgramData\anaconda3\envs\llms\Lib\site-packages\trl\trainer\sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [ ]:
trainer.train()


Step,Training Loss
500,2.536500
1000,2.319600
1500,2.279500
2000,2.282000
2500,2.242800
3000,2.237900
3500,2.238400
4000,2.261800
4500,2.236100
5000,2.229800
